# Clasificación — Predicción de Dirección del Oro

**Autor**: Juan (Scrum Master)

**Objetivo**: Predecir si el oro sube o baja al día siguiente usando variables de precio e indicadores macro.

**Dataset**: GC=F (oro) + DXY (dólar) + VIX (volatilidad) + TNX (bono 10Y) desde 2015.
Datos descargados por Joel mediante `src/extract/extract.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost no instalado. Se omitirá.')

sns.set_style('whitegrid')
%matplotlib inline

## 1. Carga de datos

Cargamos el CSV generado por Joel. Contiene Open, High, Low, Close, Volume para cada uno de los 4 activos.

**Por qué estos tickers**:
- GC=F: precio del oro (nuestro target)
- DXY: índice del dólar. El oro se mueve inversamente al dólar
- VIX: índice de miedo. En crisis el oro sube como activo refugio
- TNX: bono USA a 10 años. Tipos altos → oro menos atractivo

In [ ]:
df = pd.read_csv('data/raw/gold-macro-data.csv', index_col='Date', parse_dates=True)

print(f'Registros: {len(df)}')
print(f'Rango: {df.index[0].date()} a {df.index[-1].date()}')
print(f'Columnas: {df.columns.tolist()}')
df.head()

In [ ]:
df.describe()

### 1.1 Limpieza de columnas

El CSV tiene nombres de columna con paréntesis y comillas. Los limpiamos para trabajar cómodamente.

In [ ]:
# Limpiar nombres de columna: quitar prefijos y paréntesis
df.columns = [
    col.replace("('", '_').replace("', '", '_').replace("')", '')
    for col in df.columns
]

# Extraer solo las columnas de cierre de cada activo
close_cols = [c for c in df.columns if 'Close' in c]
print('Columnas de cierre:', close_cols)

data = df[close_cols].copy()
data.columns = ['gold', 'dxy', 'vix', 'tnx']
data.dropna(inplace=True)

print(f'Registros limpios: {len(data)}')
data.head()

### 1.2 Visualización rápida

Evolución de las 4 series. Se observa la relación inversa entre oro y dólar, y los picos de VIX en momentos de crisis (2020).

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
titles = ['Oro (GC=F)', 'Dólar (DXY)', 'Volatilidad (VIX)', 'Bono 10Y (TNX)']
for ax, col, title in zip(axes, ['gold', 'dxy', 'vix', 'tnx'], titles):
    ax.plot(data.index, data[col], linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 2. Feature engineering

Creamos targets y features:
- **Target binario**: 1 si el oro sube mañana, 0 si baja
- **Target multiclase**: 2 (sube fuerte >1%), 1 (estable), 0 (baja fuerte <-1%)
- **Features técnicas**: retornos, rango, medias móviles, RSI, MACD, volatilidad
- **Macros**: DXY, VIX, TNX en bruto y sus retornos diarios

In [ ]:
d = data.copy()

# Targets
d['target_bin'] = (d['gold'].shift(-1) > d['gold']).astype(int)
retornos_futuros = d['gold'].pct_change(-1) * 100
d['target_multi'] = pd.cut(
    retornos_futuros,
    bins=[-float('inf'), -1, 1, float('inf')],
    labels=[0, 1, 2]
).astype(int)

# Features de precio del oro
d['returns'] = d['gold'].pct_change()
for period in [5, 10, 21]:
    d[f'ma_{period}'] = d['gold'].rolling(period).mean()
    d[f'volatility_{period}'] = d['returns'].rolling(period).std()
d['close_ma5_ratio'] = d['gold'] / d['ma_5']

# RSI
def rsi(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

d['rsi'] = rsi(d['gold'])

# MACD
ema_12 = d['gold'].ewm(span=12).mean()
ema_26 = d['gold'].ewm(span=26).mean()
d['macd'] = ema_12 - ema_26
d['macd_signal'] = d['macd'].ewm(span=9).mean()

# Features macro
for col in ['dxy', 'vix', 'tnx']:
    d[f'{col}_return'] = d[col].pct_change()

d.dropna(inplace=True)

print(f'Registros finales: {len(d)}')
print(f'\nDistribución target binario:')
print(d['target_bin'].value_counts(normalize=True))
print(f'\nDistribución target multiclase:')
print(d['target_multi'].value_counts(normalize=True))

## 3. Preprocesamiento

Dividimos en train (80%) y test (20%) manteniendo orden temporal. El scaler se ajusta solo en train para evitar data leakage.

In [ ]:
feature_cols = [
    'returns', 'ma_5', 'ma_10', 'ma_21',
    'volatility_5', 'volatility_10', 'volatility_21',
    'close_ma5_ratio', 'rsi', 'macd', 'macd_signal',
    'dxy_return', 'vix_return', 'tnx_return',
    'dxy', 'vix', 'tnx'
]

X = d[feature_cols]
y_bin = d['target_bin']
y_multi = d['target_multi']

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_bin_train, y_bin_test = y_bin.iloc[:split_idx], y_bin.iloc[split_idx:]
y_multi_train, y_multi_test = y_multi.iloc[:split_idx], y_multi.iloc[split_idx:]

print(f'Train: {len(X_train)} ({X_train.index[0].date()} a {X_train.index[-1].date()})')
print(f'Test:  {len(X_test)} ({X_test.index[0].date()} a {X_test.index[-1].date()})')

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## 4. Modelos

Entrenamos 4 modelos:
1. **DummyClassifier** (baseline): clase mayoritaria
2. **Logistic Regression**: lineal, interpretable
3. **Random Forest**: ensemble robusto
4. **XGBoost**: gradient boosting (si instalado)

In [ ]:
models = {
    'Dummy': DummyClassifier(strategy='most_frequent', random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
}

if XGB_AVAILABLE:
    models['XGBoost'] = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        eval_metric='logloss', random_state=42
    )

### 4.1 Clasificación binaria

In [ ]:
results_bin = []

for name, model in models.items():
    model.fit(X_train_s, y_bin_train)
    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:, 1] if hasattr(model, 'predict_proba') else None
    
    results_bin.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_bin_test, y_pred),
        'Precision': precision_score(y_bin_test, y_pred, zero_division=0),
        'Recall': recall_score(y_bin_test, y_pred, zero_division=0),
        'F1': f1_score(y_bin_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_bin_test, y_proba) if y_proba is not None else None
    })

df_results_bin = pd.DataFrame(results_bin)
df_results_bin

#### Interpretación

Comparamos contra el baseline (Dummy). Si no lo superan, no hay señal predictiva. Si lo superan, el modelo aporta valor sobre el azar.

In [ ]:
dummy_acc = df_results_bin.loc[df_results_bin['Modelo'] == 'Dummy', 'Accuracy'].values[0]
print(f'Baseline (siempre clase mayoritaria): {dummy_acc:.2%}')
print()
for _, row in df_results_bin.iterrows():
    if row['Modelo'] != 'Dummy':
        diff = row['Accuracy'] - dummy_acc
        signo = '+' if diff > 0 else ''
        print(f"{row['Modelo']:25s} → Accuracy: {row['Accuracy']:.2%} ({signo}{diff:.2%} vs baseline, F1: {row['F1']:.3f})")

### 4.2 Clasificación multiclase

3 categorías: sube fuerte (>1%), estable, baja fuerte (<-1%).

In [ ]:
results_multi = []

for name, model in models.items():
    m = model.__class__(**model.get_params()) if hasattr(model, 'get_params') else model
    m.fit(X_train_s, y_multi_train)
    y_pred = m.predict(X_test_s)
    
    results_multi.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_multi_test, y_pred),
        'F1 (weighted)': f1_score(y_multi_test, y_pred, average='weighted', zero_division=0)
    })

df_results_multi = pd.DataFrame(results_multi)
df_results_multi

### 4.3 Backtesting

Simulación de trading: cuando el modelo predice UP, ganamos el retorno del día. Si DOWN, ganamos 0 (nos quedamos fuera).

Comparamos con comprar y mantener (buy & hold).

In [ ]:
retornos_test = d['returns'].iloc[split_idx:].values

resultados_backtest = []

for name, model in models.items():
    model.fit(X_train_s, y_bin_train)
    y_pred = model.predict(X_test_s)
    
    retornos_estrategia = retornos_test * y_pred
    retorno_acumulado = np.prod(1 + retornos_estrategia) - 1
    retorno_bh = np.prod(1 + retornos_test) - 1
    
    dias_operados = (y_pred == 1).sum()
    aciertos = ((y_pred == 1) & (retornos_test > 0)).sum()
    win_rate = aciertos / dias_operados if dias_operados > 0 else 0
    
    resultados_backtest.append({
        'Modelo': name,
        'Retorno estrategia': f'{retorno_acumulado:.2%}',
        'Retorno buy & hold': f'{retorno_bh:.2%}',
        'Diferencia vs BH': f'{retorno_acumulado - retorno_bh:+.2%}',
        'Win rate': f'{win_rate:.2%}',
        'Días operados': dias_operados
    })

df_backtest = pd.DataFrame(resultados_backtest)
df_backtest

#### Interpretación del backtesting

Si la estrategia del modelo gana a buy & hold, el modelo tiene valor real. Si pierde, las predicciones no son lo suficientemente precisas para generar rentabilidad.

Un win rate >50% con suficientes operaciones indica señal predictiva.

## 5. Detección de overfitting

Comparamos rendimiento en train vs test. Diferencia >10% sugiere sobreajuste.

In [ ]:
print('Comparación Train vs Test (Accuracy):')
print()
for name, model in models.items():
    train_acc = accuracy_score(y_bin_train, model.predict(X_train_s))
    test_acc = accuracy_score(y_bin_test, model.predict(X_test_s))
    diff = train_acc - test_acc
    warning = '⚠️ Posible overfitting' if diff > 0.1 else '✅ OK'
    print(f"{name:25s} Train: {train_acc:.2%} | Test: {test_acc:.2%} | Dif: {diff:+.2%} {warning}")

In [ ]:
if 'Random Forest' in models:
    cv_scores = cross_val_score(
        models['Random Forest'], X_train_s, y_bin_train,
        cv=5, scoring='f1'
    )
    print(f'Random Forest - CV F1: {cv_scores}')
    print(f'Media: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## 6. Importancia de features

In [ ]:
if 'Random Forest' in models:
    rf = models['Random Forest']
    importancias = pd.DataFrame({
        'Feature': feature_cols,
        'Importancia': rf.feature_importances_
    }).sort_values('Importancia', ascending=False)

    plt.figure(figsize=(8, 6))
    sns.barplot(data=importancias.head(10), x='Importancia', y='Feature')
    plt.title('Top 10 Features - Random Forest')
    plt.tight_layout()
    plt.show()
    
    print('Feature importances:')
    for _, row in importancias.head(10).iterrows():
        print(f"  {row['Feature']:20s} {row['Importancia']:.4f}")

## 7. Conclusiones

| Aspecto | Resultado |
|---------|-----------|
| **Binaria** | ¿Los modelos superan al baseline? |
| **Multiclase** | ¿Hay señal para 3 categorías? |
| **Backtesting** | ¿La estrategia gana a buy & hold? |
| **Overfitting** | ¿Train vs test consistentes? |
| **Features clave** | ¿Qué variables pesan más? |

### Mejoras posibles

1. Añadir CPI y Fed Funds Rate desde FRED
2. Horizonte semanal (más señal, menos ruido)
3. GridSearchCV para ajustar hiperparámetros
4. Estrategia con umbral de confianza (>70% para operar)